# Day 2 — QAT Pareto sweep

Run this notebook once per available GPU. In each copy, set `RUN_NAME`, `WEIGHT_BITS`, and `ACTIVATION_BITS` to one of W8A8, W6A6, W4A6, or W4A4. Each command reloads the immutable baseline and logs complete console output with `tee`.


In [1]:
import os
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret('GITHUB_TOKEN')
username, repo_name = 'AdiGiriIIT', 'CS6886--Assignment-2'
!git clone https://{token}@github.com/{username}/{repo_name}.git assignment-2
%cd assignment-2


Cloning into 'assignment-2'...
remote: Enumerating objects: 73, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 73 (delta 21), reused 67 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (73/73), 899.94 KiB | 5.73 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/kaggle/working/assignment-2


In [2]:
from pathlib import Path

BASELINE_SOURCE = '/kaggle/input/datasets/adityagirishep23b048/baseline/baseline.pt'  # adjust only if your dataset mount differs
DATA_DIR = '/kaggle/input/datasets/adityagirishep23b048/cifar-data'
EXPECTED_SHA256 = '02ff38ac832c9fa3d72cad3db375b1103991b64eaf417c796ab352b49fbaae3d'

!python -m pip install -q PyYAML matplotlib
!mkdir -p results/checkpoints results/logs experiments/sweeps
!cp $BASELINE_SOURCE results/checkpoints/baseline.pt
!sha256sum results/checkpoints/baseline.pt
cifar_dir = Path(DATA_DIR) / 'cifar-10-batches-py'
assert all((cifar_dir / name).is_file() for name in ['data_batch_1', 'data_batch_2', 'data_batch_3', 'data_batch_4', 'data_batch_5', 'test_batch', 'batches.meta'])
print('Using CIFAR-10 at', cifar_dir)


02ff38ac832c9fa3d72cad3db375b1103991b64eaf417c796ab352b49fbaae3d  results/checkpoints/baseline.pt
Using CIFAR-10 at /kaggle/input/datasets/adityagirishep23b048/cifar-data/cifar-10-batches-py


In [3]:
# Required correctness gates before consuming a full GPU run. pipefail preserves failures through tee.
!set -o pipefail; python -m unittest discover -s tests -v 2>&1 | tee results/logs/day2-correctness.log
!set -o pipefail; python -m src.evaluate --checkpoint results/checkpoints/baseline.pt --data-dir "$DATA_DIR" --device cuda 2>&1 | tee results/logs/day2-baseline-reconstruction.log
!set -o pipefail; nvidia-smi 2>&1 | tee results/logs/day2-gpu.log


test_fold_equivalence_and_accounting (test_compression.CompressionTests.test_fold_equivalence_and_accounting) ... ok
test_pack_round_trip_and_padding (test_compression.CompressionTests.test_pack_round_trip_and_padding) ... ok
test_ranges_and_levels (test_compression.CompressionTests.test_ranges_and_levels) ... ok
test_residual_add_uses_one_signed_scale_and_transition (test_compression.CompressionTests.test_residual_add_uses_one_signed_scale_and_transition) ... ok
test_scale_positive_and_per_channel (test_compression.CompressionTests.test_scale_positive_and_per_channel) ... ok
test_ste_and_clipping_gradient (test_compression.CompressionTests.test_ste_and_clipping_gradient) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.524s

OK
Checkpoint: results/checkpoints/baseline.pt
Test loss: 0.2401
Test top-1 accuracy: 92.69%
Fri Sep  4 18:36:18 2026       
+-----------------------------------------------------------------------------------------+


In [4]:
# Change these three values in each parallel notebook.
RUN_NAME = 'w8a8-seed6886'
WEIGHT_BITS = 8
ACTIVATION_BITS = 8
EPOCHS = 12

assert (WEIGHT_BITS, ACTIVATION_BITS) in {(8, 8), (6, 6), (4, 6), (4, 4)}
print(f'Launching {RUN_NAME}: W{WEIGHT_BITS}A{ACTIVATION_BITS}')


Launching w8a8-seed6886: W8A8


In [5]:
# pipefail makes a failed training command fail the cell even though tee is used.
!set -o pipefail; python -m src.qat --checkpoint results/checkpoints/baseline.pt --data-dir "{DATA_DIR}" --device cuda --weight-bits {WEIGHT_BITS} --activation-bits {ACTIVATION_BITS} --epochs {EPOCHS} --run-name {RUN_NAME} 2>&1 | tee results/logs/{RUN_NAME}.log


epoch=01/12 W8A8 train=96.29% test=92.03% loss=0.2602 elapsed=49.6s
epoch=02/12 W8A8 train=96.54% test=92.22% loss=0.2506 elapsed=49.3s
epoch=03/12 W8A8 train=96.84% test=92.32% loss=0.2526 elapsed=48.2s
epoch=04/12 W8A8 train=97.34% test=92.38% loss=0.2503 elapsed=47.7s
epoch=05/12 W8A8 train=97.50% test=92.21% loss=0.2584 elapsed=48.3s
epoch=06/12 W8A8 train=97.73% test=91.89% loss=0.2606 elapsed=48.2s
epoch=07/12 W8A8 train=97.89% test=92.09% loss=0.2556 elapsed=49.0s
epoch=08/12 W8A8 train=98.11% test=92.23% loss=0.2522 elapsed=48.2s
epoch=09/12 W8A8 train=98.21% test=92.43% loss=0.2556 elapsed=50.2s
epoch=10/12 W8A8 train=98.51% test=92.38% loss=0.2738 elapsed=50.0s
epoch=11/12 W8A8 train=98.46% test=92.47% loss=0.2753 elapsed=49.9s
epoch=12/12 W8A8 train=98.48% test=92.39% loss=0.2738 elapsed=49.6s
best_test_accuracy=92.47% weight_ratio=3.568x weight_bytes=2545456 run_record=experiments/sweeps/w8a8-seed6886


In [ ]:
from pathlib import Path
import hashlib
import tarfile
from IPython.display import FileLink

run_dir = Path("experiments/sweeps") / RUN_NAME
paths = [
    run_dir,
    Path("results/logs") / f"{RUN_NAME}.log",
    Path("results/checkpoints") / f"qat-{RUN_NAME}-best.pt",
    Path("results/checkpoints") / f"qat-{RUN_NAME}-latest.pt",
]
paths = [path for path in paths if path.exists()]

if not (run_dir / "metrics.json").exists():
    raise FileNotFoundError(f"Missing {run_dir}/metrics.json")
if len(paths) < 3:
    raise FileNotFoundError("Expected run record, log, and at least one QAT checkpoint.")

for path in paths:
    if path.is_file():
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        print(f"{digest}  {path}")

archive = Path(f"{RUN_NAME}-artifacts.tgz")
with tarfile.open(archive, "w:gz") as tar:
    for path in paths:
        tar.add(path, arcname=str(path))

print(f"Created {archive} ({archive.stat().st_size:,} bytes)")
FileLink(str(archive))

{
  "run_name": "w8a8-seed6886",
  "baseline_sha256": "02ff38ac832c9fa3d72cad3db375b1103991b64eaf417c796ab352b49fbaae3d",
  "weight_bits": 8,
  "activation_bits": 8,
  "edge_bits": null,
  "epochs": 12,
  "best_test_accuracy": 92.47,
  "packed_weight_bytes": 2202560,
  "weight_scale_bytes": 68264,
  "bias_bytes": 272936,
  "descriptor_bytes": 1696,
  "compressed_weight_bytes": 2545456,
  "fp32_weight_bytes": 9083176,
  "weight_compression_ratio": 3.56838853234941,
  "best_checkpoint": "results/checkpoints/qat-w8a8-seed6886-best.pt",
  "latest_checkpoint": "results/checkpoints/qat-w8a8-seed6886-latest.pt"
}
375d352eeb0436729d1e96fe7ad8a39b63547ba985be603f1485c093a5229fc6  results/checkpoints/qat-w8a8-seed6886-best.pt
0273fb409e520f0215072da4d7a1b32ae85483087564d512e608cdf5d9c9a159  results/checkpoints/qat-w8a8-seed6886-latest.pt
tar: results/logs/.log: Cannot stat: No such file or directory
tar: results/checkpoints/qat--best.pt: Cannot stat: No such file or directory
tar: Exiting with f

/kaggle/working/assignment-2/w8a8-seed6886-artifacts.tgz